# Running AlphaFold

## Make sure imports work

In [1]:
import os, sys, json, pathlib, time
import numpy as np
import torch
import matplotlib.pyplot as plt

print("python     ", sys.version.split()[0])
print("torch      ", torch.__version__)
print("cuda avail ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device     ", torch.cuda.get_device_name(0))
    print("vram (GB)  ", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

import openfold
print("openfold   ", pathlib.Path(openfold.__file__).parent)
import attn_core_inplace_cuda
print("cuda kernel  OK")

python      3.10.20
torch       2.14.0+cu130
cuda avail  True
device      NVIDIA GeForce RTX 4060 Laptop GPU
vram (GB)   8.2
openfold    /home/caitlinlewis/repos/openfold/openfold
cuda kernel  OK


## Get protein info

In [2]:
import requests
from pathlib import Path

ACCESSIONS = {
    "myoglobin":  "P02185",  # sperm whale Mb, the 1MBN classic
    "GFP":        "P42212",  # Aequorea victoria avGFP
    "TIM":        "P00940",  # chicken triosephosphate isomerase, 1TIM
    "ubiquitin":  "P0CG48",  # human polyubiquitin-C
    "calmodulin": "P0DP23",  # human CALM1
}

out = Path("work/fasta"); out.mkdir(exist_ok=True)

for name, acc in ACCESSIONS.items():
    r = requests.get(f"https://rest.uniprot.org/uniprotkb/{acc}.fasta", timeout=30)
    r.raise_for_status()
    header, *seq = r.text.strip().split("\n")
    seq = "".join(seq)
    print(f"{name:12s} {acc}  len={len(seq):4d}  {header[:70]}")
    (out / f"{name}.fasta").write_text(f">{name}\n{seq}\n")

myoglobin    P02185  len= 154  >sp|P02185|MYG_PHYMC Myoglobin OS=Physeter macrocephalus OX=9755 GN=MB
GFP          P42212  len= 238  >sp|P42212|GFP_AEQVI Green fluorescent protein OS=Aequorea victoria OX
TIM          P00940  len= 248  >sp|P00940|TPIS_CHICK Triosephosphate isomerase OS=Gallus gallus OX=90
ubiquitin    P0CG48  len= 685  >sp|P0CG48|UBC_HUMAN Polyubiquitin-C OS=Homo sapiens OX=9606 GN=UBC PE
calmodulin   P0DP23  len= 149  >sp|P0DP23|CALM1_HUMAN Calmodulin-1 OS=Homo sapiens OX=9606 GN=CALM1 P


### Manually edit to get smaller chunk of ubiquitin

In [3]:
p = out / "ubiquitin.fasta"
seq = "".join(p.read_text().split("\n")[1:])
p.write_text(f">ubiquitin\n{seq[:76]}\n")
assert seq[:76].endswith("LRGG")  # canonical C-terminal tail

## Config

In [4]:
MODEL_NAME = "model_3_ptm"

# per-protein annotations for the viz; only calmodulin has lobes
ANNOTATIONS = {
    "calmodulin": {
        "N_LOBE":   (1, 75),
        "LINKER":   (76, 81),
        "C_LOBE":   (82, 148),
        "EF_HANDS": [(20, 31), (56, 67), (93, 104), (129, 140)],
    },
}

OPENFOLD_DIR = pathlib.Path(os.environ.get("OPENFOLD_DIR", "~/repos/openfold")).expanduser()
PARAMS_PATH  = OPENFOLD_DIR / "openfold" / "resources" / "params" / f"params_{MODEL_NAME}.npz"

WORK = pathlib.Path("./work").resolve()
for sub in ["fasta", "alignments", "out", "ref", "curves", "pair"]:
    (WORK / sub).mkdir(parents=True, exist_ok=True)

NAMES = list(ACCESSIONS.keys())

SEQUENCES = {}
for name in NAMES:
    txt = (WORK / "fasta" / f"{name}.fasta").read_text()
    SEQUENCES[name] = "".join(txt.strip().split("\n")[1:])

print({k: len(v) for k, v in SEQUENCES.items()})
print("params exists:", PARAMS_PATH.exists())

{'myoglobin': 154, 'GFP': 238, 'TIM': 248, 'ubiquitin': 76, 'calmodulin': 149}
params exists: True


## Get MSA alignment for each protein

In [5]:
from colabfold.colabfold import run_mmseqs2

MSA_DEPTH = {}
(WORK / "mmseqs_cache").mkdir(parents=True, exist_ok=True)
for name in NAMES:
    (WORK / "alignments" / name).mkdir(parents=True, exist_ok=True)
    a3m_path = WORK / "alignments" / name / "mmseqs.a3m"

    if not a3m_path.exists():
        res = run_mmseqs2(
            SEQUENCES[name],
            prefix=str(WORK / "mmseqs_cache" / name),   # per-protein cache
            use_env=True, use_filter=True, use_templates=False,
        )
        a3m_path.write_text(res[0] if isinstance(res, (list, tuple)) else res)

    n = a3m_path.read_text().count(">")
    MSA_DEPTH[name] = n
    print(f"{name:12s} depth {n:6d}")
    assert n > 100, f"{name}: MSA suspiciously shallow"

myoglobin    depth   1331
GFP          depth    743
TIM          depth  16985
ubiquitin    depth  21577
calmodulin   depth  21884


## Load the model once

In [6]:
from openfold.config import model_config
from openfold.data import data_pipeline, feature_pipeline
from openfold.model.model import AlphaFold
from openfold.utils.import_weights import import_jax_weights_
from openfold.utils.tensor_utils import tensor_tree_map
from openfold.np import protein, residue_constants

config = model_config(MODEL_NAME)
N_BLK = config.model.evoformer_stack.no_blocks
C_Z   = config.model.evoformer_stack.c_z
print(f"blocks {N_BLK}  c_z {C_Z}")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AlphaFold(config)
model.eval()
import_jax_weights_(model, str(PARAMS_PATH), version=MODEL_NAME)
model = model.to(device)

dp = data_pipeline.DataPipeline(template_featurizer=None)
fp = feature_pipeline.FeaturePipeline(config.data)

blocks 48  c_z 128


## Run the model for each protein

In [ ]:
from openfold.np import residue_constants as rc

In [8]:
SAVE_BLOCKS = list(range(0, N_BLK, 6))   # 8 of 48; see note below

pair_store = {}
def _mk_hook(i):
    def hook(mod, inp, outp):
        if i in SAVE_BLOCKS:
            z = outp[1] if isinstance(outp, (list, tuple)) else outp
            pair_store[i] = z.detach().half().cpu().numpy()
    return hook

handles = [blk.register_forward_hook(_mk_hook(i))
           for i, blk in enumerate(model.evoformer.blocks)]

SUMMARY = {}
for name in NAMES:
    seq = SEQUENCES[name]
    L = len(seq)
    pair_store.clear()

    feature_dict = dp.process_fasta(
        fasta_path=str(WORK / "fasta" / f"{name}.fasta"),
        alignment_dir=str(WORK / "alignments" / name),
    )
    processed = fp.process_features(feature_dict, mode="predict")
    batch = {k: torch.as_tensor(v, device=device) for k, v in processed.items()}

    t0 = time.time()
    with torch.no_grad():
        out_t = model(batch)
    dt = time.time() - t0
    out = tensor_tree_map(lambda x: x.detach().cpu().numpy(), out_t)
    del out_t

    plddt = out["plddt"]                     # (L,)
    plddt_b = np.repeat(plddt[:, None], residue_constants.atom_type_num, axis=-1)

    batch_last = {k: v[..., -1].cpu().numpy() for k, v in batch.items()}
    unrelaxed = protein.from_prediction(
        features=batch_last, result=out, b_factors=plddt_b,
        remove_leading_feature_dimension=False,
    )
    pdb_path = WORK / "out" / f"{name}_{MODEL_NAME}.pdb"
    pdb_path.write_text(protein.to_pdb(unrelaxed))

    n_ca = sum(1 for l in pdb_path.read_text().splitlines()
               if l.startswith("ATOM") and l[12:16].strip() == "CA")
    assert n_ca == L, f"{name}: expected {L} CA, got {n_ca}"

    BACKBONE = ["N", "CA", "C", "O"]
    resnames = np.array([rc.restype_1to3.get(rc.restypes[i], "UNK")
                     for i in batch_last["aatype"]])

    # arrays for the viz
    np.savez_compressed(
        WORK / "out" / f"{name}_{MODEL_NAME}_scalars.npz",
        plddt=plddt.astype(np.float32),
        pae=out.get("predicted_aligned_error", np.zeros((L, L), np.float32)).astype(np.float32),
        ptm=np.asarray(out.get("ptm_score", np.nan), np.float32),
        msa=feature_dict["msa"].astype(np.uint8),
        deletion=feature_dict["deletion_matrix_int"].astype(np.uint8),
        seq=np.frombuffer(seq.encode(), np.uint8),
        atom_positions=out["final_atom_positions"].astype(np.float32),  
        atom_mask=out["final_atom_mask"].astype(bool),                  
        aatype=batch_last["aatype"].astype(np.uint8),                   
        residue_index=batch_last["residue_index"].astype(np.int32),
        resname=resnames,                                        # (L,) '<U3'
        bb_index=np.array([rc.atom_order[a] for a in BACKBONE]), # (4,)
        bb_names=np.array(BACKBONE),
        )
    np.savez(
        WORK / "pair" / f"{name}_{MODEL_NAME}_z.npz",
        blocks=np.asarray(SAVE_BLOCKS),
        **{f"z_{i:02d}": pair_store[i] for i in SAVE_BLOCKS},
    )

    SUMMARY[name] = dict(L=L, plddt=float(plddt.mean()),
                         depth=MSA_DEPTH[name], secs=round(dt, 1))
    print(f"{name:12s} L={L:4d}  pLDDT {plddt.mean():5.1f}  {dt:5.1f}s")

    del batch, out, processed, feature_dict
    torch.cuda.empty_cache()

for h in handles:
    h.remove()

/home/caitlinlewis/repos/openfold/openfold/data/data_transforms.py:1261: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:356.)
  protein[k] = v[slices]
W0912 17:27:34.834000 73654 torch/fx/_symbolic_trace.py:57] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
/home/caitlinlewis/repos/openfold/openfold/model/triangular_multiplicative_update.py:373: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tup

myoglobin    L= 154  pLDDT  95.8   34.0s


[W912 17:28:09.521501017 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 3720347648 bytes (free: 1665662976, total: 8186822656).
[W912 17:28:09.720818895 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 3720347648 bytes (free: 1692925952, total: 8186822656).
[W912 17:28:14.355635666 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1996488704 bytes (free: 1690828800, total: 8186822656).


GFP          L= 238  pLDDT  95.3   63.1s


[W912 17:29:15.775615249 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 4039114752 bytes (free: 1248329728, total: 8186822656).
[W912 17:29:16.034252063 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 4039114752 bytes (free: 1275592704, total: 8186822656).
[W912 17:29:21.708726478 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2080374784 bytes (free: 1273495552, total: 8186822656).
[W912 17:29:23.184391899 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2080374784 bytes (free: 1984430080, total: 8186822656).
[W912 17:29:23.478145441 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2080374784 bytes (free: 1982332928, total: 8186822656).
[W912 17:29:23.599790164 CUDACachingAllocator.cpp:3934] memory allocation failed with

TIM          L= 248  pLDDT  96.3   69.3s
ubiquitin    L=  76  pLDDT  95.5   13.8s
calmodulin   L= 149  pLDDT  80.8   33.3s
